In [1]:
# Étape 1 : Importer les bibliothèques nécessaires et charger les données

import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration d'affichage
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

# Charger les données
df = pd.read_csv("C:/Users/ayadi/OneDrive - ESPRIT (1)/Bureau/pa bi/data_externe/most_used_beauty_cosmetics_products_extended.csv")

# Aperçu des données
print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")
df.head()


C:\Users\ayadi\anaconda3\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Nombre de lignes : 15000
Nombre de colonnes : 14


,Product_Name,Brand,Category,Usage_Frequency,Price_USD,Rating,Number_of_Reviews,Product_Size,Skin_Type,Gender_Target,Packaging_Type,Main_Ingredient,Cruelty_Free,Country_of_Origin
0,Ultra Face Mask,Drunk Elephant,Blush,Weekly,67.85,1.4,686,30ml,Sensitive,Female,Tube,Retinol,False,Australia
1,Ultra Lipstick,Laura Mercier,Makeup Remover,Occasional,116.43,4.2,5483,250ml,Dry,Unisex,Bottle,Shea Butter,False,UK
2,Ultra Serum,Natasha Denona,Highlighter,Daily,90.84,1.6,5039,100ml,Sensitive,Male,Compact,Aloe Vera,True,Italy
3,Divine Serum,Ilia Beauty,Face Mask,Occasional,55.17,3.2,6202,250ml,Normal,Male,Tube,Glycerin,True,South Korea
4,Super Foundation,Charlotte Tilbury,Highlighter,Occasional,140.56,1.7,297,100ml,Oily,Female,Compact,Glycerin,False,Germany


In [7]:
# Étape 2 : Analyse exploratoire simple

# Résumé statistique
print("Résumé statistique des colonnes numériques :")
print(df.describe())

# Informations sur les types et valeurs manquantes
print("\nInformations générales :")
print(df.info())

# Vérification des valeurs uniques pour les colonnes catégorielles
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\nValeurs uniques dans '{col}':")
    print(df[col].value_counts())


Résumé statistique des colonnes numériques :
          Price_USD        Rating  Number_of_Reviews
count  15000.000000  15000.000000       15000.000000
mean      80.134108      3.002327        5014.231333
std       40.402983      1.168029        2855.665464
min       10.000000      1.000000          52.000000
25%       45.480000      2.000000        2562.000000
50%       80.040000      3.000000        5002.000000
75%      114.760000      4.000000        7497.000000
max      149.990000      5.000000       10000.000000

Informations générales :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Product_Name       15000 non-null  object 
 1   Brand              15000 non-null  object 
 2   Category           15000 non-null  object 
 3   Usage_Frequency    15000 non-null  object 
 4   Price_USD          15000 non-null  float64
 5   Ra

In [10]:
# Étape 3 : Vectorisation des caractéristiques combinées et calcul de similarité

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Création du vecteur TF-IDF à partir de la colonne "combined_features"
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

# Calcul de la similarité cosinus entre tous les produits
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Création d'une série pour accéder rapidement à l'index d'un produit par son nom
indices = pd.Series(df.index, index=df['Product_Name'])

# Fonction de recommandation corrigée
def get_recommendations(product_name, cosine_sim=cosine_sim, top_n=5):
    # Gestion des doublons : on prend le premier produit correspondant
    idx = indices[product_name]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    # Récupération des scores de similarité
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Tri des produits selon la similarité
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Sélection des top produits (hors le produit lui-même)
    sim_scores = sim_scores[1:top_n+1]

    # Récupération des index des produits recommandés
    product_indices = [i[0] for i in sim_scores]

    # Retour des produits recommandés
    return df[['Product_Name', 'Brand', 'Category', 'Skin_Type', 'Rating']].iloc[product_indices]


In [11]:
get_recommendations("Magic Face Oil")


,Product_Name,Brand,Category,Skin_Type,Rating
2206,Magic Makeup Remover,Laura Mercier,Face Oil,Sensitive,2.7
4544,Divine Lip Gloss,Laura Mercier,Face Oil,Sensitive,3.4
6833,Perfect Eye Shadow,Laura Mercier,Face Oil,Sensitive,2.9
2708,Divine Eye Shadow,Laura Mercier,Setting Spray,Sensitive,4.0
7843,Divine Serum,Laura Mercier,Lip Gloss,Sensitive,4.7


In [12]:
# Étape 4 : Interface interactive avec menu déroulant

from IPython.display import display, clear_output
import ipywidgets as widgets

# Création du menu déroulant avec les noms de produits uniques (triés)
product_dropdown = widgets.Dropdown(
    options=sorted(df['Product_Name'].unique()),
    description='Produit:',
    layout=widgets.Layout(width='50%')
)

# Fonction appelée lors de la sélection
def on_product_selected(change):
    clear_output(wait=True)
    display(product_dropdown)
    print(f"\n🔎 Recommandations pour : {change.new}\n")
    display(get_recommendations(change.new))

# Lier la fonction à l'événement de sélection
product_dropdown.observe(on_product_selected, names='value')

# Afficher le menu déroulant
display(product_dropdown)


Dropdown(description='Produit:', index=30, layout=Layout(width='50%'), options=('Divine BB Cream', 'Divine Blu…

Dropdown(description='Peau:', index=4, layout=Layout(width='50%'), options=('Combination', 'Dry', 'Normal', 'O…

Dropdown(description='Usage:', index=1, layout=Layout(width='50%'), options=('Daily', 'Monthly', 'Occasional',…


🔍 Recommandations pour : Magic Contour | Peau : Sensitive | Usage : Monthly



,Product_Name,Brand,Category,Skin_Type,Usage_Frequency,Rating
13549,Super Lip Gloss,Kylie Cosmetics,Powder,Sensitive,Monthly,1.6
7407,Super Lip Gloss,Kylie Cosmetics,Powder,Sensitive,Monthly,3.6
10131,Divine Eye Shadow,Kylie Cosmetics,Face Oil,Sensitive,Monthly,2.7
12857,Magic Mascara,Kylie Cosmetics,Concealer,Sensitive,Monthly,4.4
2288,Divine Mascara,Kylie Cosmetics,Makeup Remover,Sensitive,Monthly,2.3


In [13]:
# Étape 5 : Fonction de recommandation avec filtre utilisateur

def get_filtered_recommendations(product_name, user_skin_type=None, user_usage=None, top_n=5):
    # Obtenir l’index du produit
    idx = indices[product_name]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    # Scores de similarité
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]  # On enlève le produit lui-même

    # Appliquer le filtre selon skin type et usage
    filtered_indices = []
    for i, score in sim_scores:
        if user_skin_type and df.iloc[i]['Skin_Type'] != user_skin_type:
            continue
        if user_usage and df.iloc[i]['Usage_Frequency'] != user_usage:
            continue
        filtered_indices.append(i)
        if len(filtered_indices) >= top_n:
            break

    return df[['Product_Name', 'Brand', 'Category', 'Skin_Type', 'Usage_Frequency', 'Rating']].iloc[filtered_indices]


In [18]:
get_filtered_recommendations(
    product_name="Magic Face Oil",
    user_skin_type="Sensitive",
    user_usage="Weekly",
    top_n=5
)


,Product_Name,Brand,Category,Skin_Type,Usage_Frequency,Rating
6833,Perfect Eye Shadow,Laura Mercier,Face Oil,Sensitive,Weekly,2.9
2708,Divine Eye Shadow,Laura Mercier,Setting Spray,Sensitive,Weekly,4.0
8533,Super Makeup Remover,Laura Mercier,Face Oil,Sensitive,Weekly,1.9
431,Divine Powder,Laura Mercier,Primer,Sensitive,Weekly,4.8
9055,Magic Contour,Farsali,Face Oil,Sensitive,Weekly,4.7


In [19]:
# Étape 6 : Interface interactive complète avec filtres

from IPython.display import display, clear_output
import ipywidgets as widgets

# Widgets Dropdown
product_dropdown = widgets.Dropdown(
    options=sorted(df['Product_Name'].unique()),
    description='Produit:',
    layout=widgets.Layout(width='50%')
)

skin_dropdown = widgets.Dropdown(
    options=sorted(df['Skin_Type'].unique()),
    description='Peau:',
    layout=widgets.Layout(width='50%')
)

usage_dropdown = widgets.Dropdown(
    options=sorted(df['Usage_Frequency'].unique()),
    description='Usage:',
    layout=widgets.Layout(width='50%')
)

# Fonction de mise à jour
def update_recommendations(change=None):
    clear_output(wait=True)
    display(product_dropdown, skin_dropdown, usage_dropdown)

    produit = product_dropdown.value
    peau = skin_dropdown.value
    usage = usage_dropdown.value

    print(f"\n🔍 Recommandations pour : {produit} | Peau : {peau} | Usage : {usage}\n")
    display(get_filtered_recommendations(produit, peau, usage))

# Lier les changements aux widgets
product_dropdown.observe(update_recommendations, names='value')
skin_dropdown.observe(update_recommendations, names='value')
usage_dropdown.observe(update_recommendations, names='value')

# Affichage initial
display(product_dropdown, skin_dropdown, usage_dropdown)


Dropdown(description='Produit:', index=30, layout=Layout(width='50%'), options=('Divine BB Cream', 'Divine Blu…

Dropdown(description='Peau:', index=2, layout=Layout(width='50%'), options=('Combination', 'Dry', 'Normal', 'O…

Dropdown(description='Usage:', index=1, layout=Layout(width='50%'), options=('Daily', 'Monthly', 'Occasional',…


🔍 Recommandations pour : Magic Contour | Peau : Normal | Usage : Monthly



,Product_Name,Brand,Category,Skin_Type,Usage_Frequency,Rating
3015,Magic Serum,Kylie Cosmetics,Powder,Normal,Monthly,3.1
8592,Magic Lip Gloss,Kylie Cosmetics,Powder,Normal,Monthly,4.0
6660,Divine Exfoliator,IT Cosmetics,Serum,Normal,Monthly,4.9
14783,Magic Eye Shadow,Kylie Cosmetics,Cleanser,Normal,Monthly,3.5
6976,Ultra Powder,Kylie Cosmetics,Highlighter,Normal,Monthly,4.7


In [22]:
pip install streamlit



  Obtaining dependency information for streamlit from https://files.pythonhosted.org/packages/eb/17/fc425e1d4d86e31b2aaf0812a2ef2163763a0670d671720c7c36e8679323/streamlit-1.44.1-py3-none-any.whl.metadata
  Obtaining dependency information for altair<6,>=4.0 from https://files.pythonhosted.org/packages/aa/f3/0b6ced594e51cc95d8c1fc1640d3623770d01e4969d29c0bd09945fafefa/altair-5.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for blinker<2,>=1.0.0 from https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for cachetools<6,>=4.0 from https://files.pythonhosted.org/packages/72/76/20fa66124dbe6be5cafeb312ece67de6b61dd91a0247d1ea13db4ebb33c2/cachetools-5.5.2-py3-none-any.whl.metadata
  Obtaining dependency information for gitpython!=3.1.19,<4,>=3.0.7 from https://files.pythonhosted.org/packages/1d/9a/4114a9057db2f1462d5c8f8390ab7383925fe1ac012eaa424

In [21]:
# app.py

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import streamlit as st

# 1. Charger les données
df = pd.read_csv("C:/Users/ayadi/OneDrive - ESPRIT (1)/Bureau/pa bi/data_externe/most_used_beauty_cosmetics_products_extended.csv")


# 2. Vectoriser les données
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df['Product_Name'])

# 3. Fonction de recommandation
def get_filtered_recommendations(product_name, user_skin_type=None, user_usage=None, top_n=5):
    idx = indices[product_name]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]

    filtered_indices = []
    for i, _ in sim_scores:
        if user_skin_type and df.iloc[i]['Skin_Type'] != user_skin_type:
            continue
        if user_usage and df.iloc[i]['Usage_Frequency'] != user_usage:
            continue
        filtered_indices.append(i)
        if len(filtered_indices) >= top_n:
            break

    return df[['Product_Name', 'Brand', 'Category', 'Skin_Type', 'Usage_Frequency', 'Rating']].iloc[filtered_indices]

# 4. Interface Streamlit
st.set_page_config(page_title="Recommandation Cosmétiques", layout="centered")
st.title("💄 Système de Recommandation Beauté")

st.markdown("Sélectionnez un produit et vos préférences pour découvrir les recommandations personnalisées 👇")

col1, col2 = st.columns(2)

with col1:
    product_name = st.selectbox("Produit de départ", sorted(df['Product_Name'].unique()))
with col2:
    user_skin_type = st.selectbox("Type de peau", sorted(df['Skin_Type'].unique()))

user_usage = st.selectbox("Fréquence d'utilisation", sorted(df['Usage_Frequency'].unique()))

# Lancer la recommandation
if st.button("🔍 Obtenir les recommandations"):
    results = get_filtered_recommendations(product_name, user_skin_type, user_usage, top_n=5)
    st.success(f"Produits recommandés pour '{product_name}' :")
    st.dataframe(results.reset_index(drop=True))


ModuleNotFoundError: No module named 'streamlit'